<a href="https://colab.research.google.com/github/strongman2026/colab/blob/main/comfyui_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## ComfyUI Colab 启动器

**运行顺序：**

1. ⚙️ **Cell 1 - 配置区**（每次启动必跑，读取所有配置）
2. 🏗️ **Cell 2 - 基础环境**（首次或重建环境时跑）
3. 📥 **Cell 3 - HF 模型下载**（首次或缺少模型时跑）
4. 📥 **Cell 4 - Kaggle 数据集下载**（可选，从 Kaggle 拉取 Flux 主模型）
5. 🔗 **Cell 5 - 模型部署**（整理软链接，每次启动建议跑）
6. 🔌 **Cell 6 - 自定义节点**（首次或需要新节点时跑）
7. 🚀 **Cell 7 - 启动**（日常只需跑这个）

> **敏感配置**（VPS_IP、FRP_TOKEN、HF_TOKEN、KAGGLE_API_TOKEN）请通过 Colab 左侧 🔑 Secrets 面板添加，无需在代码中硬编码。

In [ ]:
# ⚙️ Cell 1 - 配置区（每次启动必跑）
# 优先从 Colab Secrets 读取敏感配置；如找不到则使用下方手动填写的默认值。
# 在 Colab 左侧点击 🔑 图标添加对应 Secret 并开启访问权限。

import os

def _get_secret(key, fallback=""):
    """优先从 Colab Secrets 读取，找不到则回退到环境变量或 fallback。"""
    try:
        from google.colab import userdata
        val = userdata.get(key)
        if val:
            return val
    except ImportError:
        pass  # 非 Colab 环境
    except Exception:
        pass  # Secret 不存在或未授权
    return os.environ.get(key, fallback)

# =====================================================
# 🔑 敏感配置（优先从 Colab Secrets 自动读取）
# 如果没有配置 Secret，可在下方 fallback 字符串中手动填写
# =====================================================
VPS_IP           = _get_secret("VPS_IP",           "")  # FRP 服务器 IP
FRP_TOKEN        = _get_secret("FRP_TOKEN",        "")  # FRP 服务器 token
HF_TOKEN         = _get_secret("HF_TOKEN",         "")  # HuggingFace Token
KAGGLE_API_TOKEN = _get_secret("KAGGLE_API_TOKEN", "")  # Kaggle API Token

# =====================================================
# 📁 路径配置（统一 Colab 环境）
# =====================================================
COMFY_DIR   = "/content/ComfyUI"
FRP_DIR     = "/content/frp"
DATASET_DIR = "/content/dataset"
MODELS_DIR  = f"{COMFY_DIR}/models"

# 将 Token 写入环境变量，供后续 shell 命令使用
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
if KAGGLE_API_TOKEN:
    os.environ["KAGGLE_API_TOKEN"] = KAGGLE_API_TOKEN

# =====================================================
# 🛒 应用商店开关 (ComfyUI-Manager)
# False = 日常跑图（隐藏 Manager，加速启动）
# True  = 需要安装/更新插件时开启
# =====================================================
ENABLE_MANAGER = False

print("✅ 配置加载完成")
print(f"   VPS_IP          : {'✅ 已配置' if VPS_IP else '⚠️ 未配置（FRP 隧道将跳过）'}")
print(f"   FRP_TOKEN       : {'✅ 已配置' if FRP_TOKEN else '⚠️ 未配置（FRP 隧道将跳过）'}")
print(f"   HF_TOKEN        : {'✅ 已配置' if HF_TOKEN else '⚠️ 未配置（访问私有 HF 模型需要）'}")
print(f"   KAGGLE_API_TOKEN: {'✅ 已配置' if KAGGLE_API_TOKEN else '⚠️ 未配置（从 Kaggle 下载时需要）'}")
print(f"   ENABLE_MANAGER  : {'True（Manager 开启）' if ENABLE_MANAGER else 'False（Manager 关闭，加速启动）'}")


In [ ]:
# 🏗️ Cell 2 - 基础环境（首次或重建时运行）
# 克隆 ComfyUI + ComfyUI-Manager，安装依赖，部署 FRP 客户端。
# 幂等：已存在则跳过，可重复运行。

import os, sys, subprocess

# ----- ComfyUI -----
if not os.path.isdir(COMFY_DIR):
    print("📥 正在克隆 ComfyUI...")
    os.system(f"git clone https://github.com/comfyanonymous/ComfyUI.git {COMFY_DIR} -q")
    print("📥 正在克隆 ComfyUI-Manager...")
    os.system(
        f"git clone https://github.com/ltdrdata/ComfyUI-Manager.git "
        f"{COMFY_DIR}/custom_nodes/ComfyUI-Manager -q"
    )
else:
    print("✅ ComfyUI 已存在，跳过克隆。")

print("📦 安装 ComfyUI 依赖...")
req = f"{COMFY_DIR}/requirements.txt"
if os.path.isfile(req):
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", req, "-q"], check=True)

# ----- FRP 客户端 -----
FRPC_BIN = f"{FRP_DIR}/frpc"
if not os.path.isfile(FRPC_BIN):
    print("🔧 正在部署 FRP 客户端...")
    os.makedirs(FRP_DIR, exist_ok=True)
    os.system(
        f"wget -q -O {FRP_DIR}/frp.tar.gz "
        "https://github.com/fatedier/frp/releases/download/v0.61.1/frp_0.61.1_linux_amd64.tar.gz"
    )
    os.system(f"tar -xzf {FRP_DIR}/frp.tar.gz -C {FRP_DIR} --strip-components=1")
    os.system(f"chmod +x {FRPC_BIN}")
else:
    print("✅ FRP 已存在，跳过。")

# ----- FRP 配置文件（仅在两个参数都配置时生成）-----
if VPS_IP and FRP_TOKEN:
    frpc_config = (
        f'serverAddr = "{VPS_IP}"\n'
        'serverPort = 7000\n'
        'auth.method = "token"\n'
        f'auth.token = "{FRP_TOKEN}"\n'
        '\n'
        '[[proxies]]\n'
        'name = "comfyui_colab"\n'
        'type = "tcp"\n'
        'localIP = "127.0.0.1"\n'
        'localPort = 8188\n'
        'remotePort = 8188\n'
    )
    with open(f"{FRP_DIR}/frpc.toml", "w") as f:
        f.write(frpc_config)
    print("✅ FRP 配置文件已写入。")
else:
    print("⚠️ VPS_IP 或 FRP_TOKEN 未配置，FRP 隧道将不可用。")

print("\n✅ Cell 2 完成！基础环境就绪。")


In [ ]:
# 📥 Cell 3 - HuggingFace 模型下载（首次或缺少模型时运行）
# 下载 Flux 所需的 VAE 和 CLIP 文本编码器。
# 已存在且大小正常则自动跳过（幂等）。

import os, subprocess

# 安装多线程下载工具（幂等）
subprocess.run(["apt-get", "-y", "install", "-qq", "aria2"],
               capture_output=True, check=True)

def aria2_download(url, dest_dir, filename, hf_token=None):
    """用 aria2c 多线程下载文件，已存在且完整则跳过。"""
    dest = os.path.join(dest_dir, filename)
    if os.path.isfile(dest) and os.path.getsize(dest) > 1024 * 1024:  # > 1 MB 视为完整
        print(f"✅ 已存在，跳过下载: {filename}")
        return
    os.makedirs(dest_dir, exist_ok=True)
    cmd = ["aria2c", "--console-log-level=error", "-c",
           "-x", "16", "-s", "16", "-k", "1M"]
    if hf_token:
        cmd.append(f"--header=Authorization: Bearer {hf_token}")
    cmd += [url, "-d", dest_dir, "-o", filename]
    print(f"🚀 下载中: {filename}")
    result = subprocess.run(cmd)
    if result.returncode == 0:
        print(f"✅ 下载完成: {filename}")
    else:
        print(f"❌ 下载失败（返回码 {result.returncode}）: {filename}")

token = HF_TOKEN or None  # 无 Token 时尝试匿名下载（公开模型可用）

# Flux VAE
aria2_download(
    "https://huggingface.co/black-forest-labs/FLUX.1-schnell/resolve/main/ae.safetensors",
    f"{MODELS_DIR}/vae", "ae.safetensors", token
)

# CLIP-L 文本编码器
aria2_download(
    "https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/clip_l.safetensors",
    f"{MODELS_DIR}/clip", "clip_l.safetensors", token
)

# T5XXL 文本编码器（约 9.8 GB）
aria2_download(
    "https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/t5xxl_fp16.safetensors",
    f"{MODELS_DIR}/clip", "t5xxl_fp16.safetensors", token
)

print("\n✅ Cell 3 完成！HF 模型下载完毕。")


In [ ]:
# 📥 Cell 4 - Kaggle 数据集下载（可选）
# 从 Kaggle 拉取 Flux 主模型到 /content/dataset/，供 Cell 5 软链接使用。
# 需要在 Cell 1 配置 KAGGLE_API_TOKEN，否则自动跳过。

import os, subprocess

if not KAGGLE_API_TOKEN:
    print("⚠️ 未配置 KAGGLE_API_TOKEN，跳过 Kaggle 下载。")
    print("   如需从 Kaggle 下载，请在 Colab Secrets 中添加 KAGGLE_API_TOKEN。")
else:
    subprocess.run(["pip", "install", "-q", "-U", "kaggle"], check=True)

    flux_path = f"{DATASET_DIR}/flux1-schnell.safetensors"
    if os.path.isfile(flux_path):
        print(f"✅ 已存在，跳过 Kaggle 下载: {flux_path}")
    else:
        os.makedirs(DATASET_DIR, exist_ok=True)
        print("⏳ 正在从 Kaggle 下载 Flux 模型...")
        subprocess.run(
            ["kaggle", "datasets", "download",
             "-d", "jarredstroman/flux-2d-assets-2026-v12",
             "-p", DATASET_DIR, "--unzip"],
            check=True
        )
        print("✅ Kaggle 数据集下载完成。")

print("\n✅ Cell 4 完成。")


In [ ]:
# 🔗 Cell 5 - 模型部署（软链接整理）
# 将下载好的模型文件软链接到 ComfyUI 对应目录。
# 幂等：可重复运行，不会因重复创建链接而报错。

import os

def safe_symlink(src, dst, desc):
    """
    安全创建软链接：
      - src == dst 时跳过（防止无效自链）
      - 自动删除已有的同名链接或文件（保证幂等）
      - 只在源文件存在时创建链接
    """
    if src == dst:
        print(f"⚠️ 跳过自链（src == dst）: {desc}")
        return
    try:
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        if os.path.islink(dst) or os.path.exists(dst):
            os.remove(dst)
        if os.path.exists(src):
            os.symlink(src, dst)
            print(f"✅ {desc}: {src} → {dst}")
        else:
            print(f"⚠️ 源文件不存在，跳过: {desc} ({src})")
    except Exception as e:
        print(f"❌ {desc} 处理失败：{e}")

# 创建 ComfyUI 标准模型目录
for d in ["unet", "clip", "vae", "ipadapter", "clip_vision", "controlnet", "checkpoints"]:
    os.makedirs(f"{MODELS_DIR}/{d}", exist_ok=True)
os.makedirs(DATASET_DIR, exist_ok=True)

# ===== Flux 主模型（从 DATASET_DIR 链接到 models/unet/）=====
safe_symlink(
    f"{DATASET_DIR}/flux1-schnell.safetensors",
    f"{MODELS_DIR}/unet/flux1-schnell.safetensors",
    "Flux 主模型 (UNet)"
)

# ===== VAE / CLIP 编码器 =====
# 说明：若通过 Cell 3 直接下载到 models/vae 和 models/clip，
# 这些文件已在正确位置，无需额外链接。
# 若你将原始文件放在 DATASET_DIR，取消下方注释并按需调整路径。
#
# safe_symlink(
#     f"{DATASET_DIR}/ae.safetensors",
#     f"{MODELS_DIR}/vae/ae.safetensors",
#     "VAE ae.safetensors"
# )
# safe_symlink(
#     f"{DATASET_DIR}/clip_l.safetensors",
#     f"{MODELS_DIR}/clip/clip_l.safetensors",
#     "CLIP-L"
# )
# safe_symlink(
#     f"{DATASET_DIR}/t5xxl_fp16.safetensors",
#     f"{MODELS_DIR}/clip/t5xxl_fp16.safetensors",
#     "T5XXL"
# )

# ===== IPAdapter 权重（可选，下载后取消注释）=====
safe_symlink(
    f"{DATASET_DIR}/flux-ip-adapter.safetensors",
    f"{MODELS_DIR}/ipadapter/flux-ip-adapter.safetensors",
    "Flux IPAdapter 权重"
)

# ===== CLIP Vision 模型（可选）=====
safe_symlink(
    f"{DATASET_DIR}/CLIP-ViT-bigG-14-laion2B-39B-b160k.safetensors",
    f"{MODELS_DIR}/clip_vision/CLIP-ViT-bigG-14-laion2B-39B-b160k.safetensors",
    "CLIP Vision 模型"
)

# ===== ControlNet 模型（可选，按需取消注释）=====
# safe_symlink(
#     f"{DATASET_DIR}/controlnet_canny.safetensors",
#     f"{MODELS_DIR}/controlnet/controlnet_canny.safetensors",
#     "ControlNet Canny"
# )

print("\n📁 当前模型目录结构：")
os.system(f"find {MODELS_DIR} -maxdepth 2 | sort")
print("\n✅ Cell 5 完成！模型目录已整理。")


In [ ]:
# 🔌 Cell 6 - 自定义节点安装（首次或需要新节点时运行）
# 安装常用自定义节点及其依赖。
# 幂等：已存在的节点跳过克隆，只安装依赖。

import os, sys, subprocess

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                "huggingface_hub"], check=True)

# 要安装的自定义节点列表：(目录名, git 仓库 URL)
CUSTOM_NODES = [
    ("ComfyUI_IPAdapter_plus",
     "https://github.com/cubiq/ComfyUI_IPAdapter_plus.git"),
    ("comfyui_controlnet_aux",
     "https://github.com/Fannovel16/comfyui_controlnet_aux.git"),
    ("rgthree-comfy",
     "https://github.com/rgthree/rgthree-comfy.git"),
]

custom_nodes_dir = f"{COMFY_DIR}/custom_nodes"
os.makedirs(custom_nodes_dir, exist_ok=True)

for name, url in CUSTOM_NODES:
    dest = os.path.join(custom_nodes_dir, name)
    if os.path.isdir(dest):
        print(f"✅ 已存在，跳过克隆: {name}")
    else:
        print(f"📥 克隆: {name}")
        ret = os.system(f"git clone {url} {dest} -q")
        if ret != 0:
            print(f"❌ 克隆失败: {name}")
            continue
    req = os.path.join(dest, "requirements.txt")
    if os.path.isfile(req):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "-r", req], check=False)

print("\n✅ Cell 6 完成！自定义节点已安装。")


In [ ]:
# 🚀 Cell 7 - 启动（日常只需跑这个）
# 停止旧进程、切换 Manager 状态、启动 ComfyUI 和 FRP 隧道。
# 非阻塞：ComfyUI 在后台运行，日志写入 /tmp/comfyui.log，Cell 运行完毕后不卡住。
# ENABLE_MANAGER 在 Cell 1 配置区中设置。

import os, time, socket, subprocess

FRPC_BIN = f"{FRP_DIR}/frpc"

# --------------------------------------------------
# 1. 停止旧进程，释放端口
# --------------------------------------------------
print("🛑 1. 停止旧进程，释放端口 8188...")
os.system("kill $(lsof -ti tcp:8188) 2>/dev/null; kill $(pgrep -f frpc) 2>/dev/null")
time.sleep(2)

# --------------------------------------------------
# 2. 切换 ComfyUI-Manager 状态
# --------------------------------------------------
MANAGER_DIR = f"{COMFY_DIR}/custom_nodes/ComfyUI-Manager"
HIDDEN_MANAGER_DIR = f"{COMFY_DIR}/.ComfyUI-Manager_disabled"
print(f"🎛️ 2. Manager: {'开启' if ENABLE_MANAGER else '关闭（加速启动）'}")
try:
    if ENABLE_MANAGER:
        if os.path.isdir(HIDDEN_MANAGER_DIR) and not os.path.isdir(MANAGER_DIR):
            os.rename(HIDDEN_MANAGER_DIR, MANAGER_DIR)
            print("   Manager 已恢复。")
    else:
        if os.path.isdir(MANAGER_DIR) and not os.path.isdir(HIDDEN_MANAGER_DIR):
            os.rename(MANAGER_DIR, HIDDEN_MANAGER_DIR)
            print("   Manager 已隐藏。")
except Exception as e:
    print(f"⚠️ Manager 状态切换失败（不影响启动）：{e}")

# --------------------------------------------------
# 3. 启动 ComfyUI（非阻塞，日志写入 /tmp/comfyui.log）
# --------------------------------------------------
print("🚀 3. 启动 ComfyUI（后台运行）...")
comfy_log = open("/tmp/comfyui.log", "w")
comfy = subprocess.Popen(
    ["python3", "main.py", "--normalvram"],
    cwd=COMFY_DIR,
    stdout=comfy_log,
    stderr=comfy_log
)

# --------------------------------------------------
# 4. 等待端口就绪（最多 120 秒）
# --------------------------------------------------
print("⏳ 4. 等待 ComfyUI 端口就绪（最多 120 秒）...")
ready = False
for i in range(120):
    time.sleep(1)
    try:
        with socket.create_connection(("127.0.0.1", 8188), timeout=1):
            ready = True
            break
    except OSError:
        pass

if not ready:
    print("❌ ComfyUI 启动超时，请检查日志：")
    os.system("tail -30 /tmp/comfyui.log")
else:
    print(f"✅ ComfyUI 已就绪！（{i + 1} 秒）")

# --------------------------------------------------
# 5. 启动 FRP 隧道（仅在 VPS_IP 和 FRP_TOKEN 均已配置时执行）
# --------------------------------------------------
frp_started = False
if VPS_IP and FRP_TOKEN and os.path.isfile(FRPC_BIN) and os.path.isfile(f"{FRP_DIR}/frpc.toml"):
    print("🌐 5. 启动 FRP 隧道...")
    frpc_log = open("/tmp/frpc.log", "w")
    frpc = subprocess.Popen(
        [FRPC_BIN, "-c", f"{FRP_DIR}/frpc.toml"],
        stdout=frpc_log,
        stderr=frpc_log
    )
    time.sleep(2)
    if frpc.poll() is None:
        frp_started = True
        print("✅ FRP 隧道已启动")
    else:
        print("❌ FRP 启动失败，请检查日志：")
        os.system("tail -20 /tmp/frpc.log")
else:
    print("⚠️ 5. FRP 未配置或文件不存在，跳过隧道启动。")

print("\n" + "=" * 55)
print(f"🎉 ComfyUI {'✅ 就绪' if ready else '❌ 启动失败'}  |  Manager: {'开启' if ENABLE_MANAGER else '已隐藏'}")
if ready:
    if frp_started:
        print(f"👉 远程访问: http://{VPS_IP}:8188")
    else:
        print("👉 本地访问: http://127.0.0.1:8188")
print("📋 ComfyUI 日志: !cat /tmp/comfyui.log")
if frp_started:
    print("📋 FRP 日志:     !cat /tmp/frpc.log")
print("=" * 55)
# 注意：不调用 comfy.wait()，Cell 运行完毕后不阻塞 Notebook。
